# Исследование индустрии видеоигр в период с 2000 по 2013 гг.

Автор: Воробьева Т.
Дата: 19.05.2026

# Цели и задачи проекта

<font color='#777778'> Провести исследование индустрии видеогр с 2000 по 2013 гг. с акцентом на RPG сегмент (от англ. Role-playing games) в разрезе основных платформ, жанров игр и полученных оценок. 
Команда игры «Секреты Темнолесья» планирует использовать результаты исследования для написания нативного контента на известном интернет-ресурсе.
</font>


# Описание данных

<font color='#777778'>В исследовании использовалось несколько основных источников данных: информация о продажах игр, а также пользовательские и экспертные оценки игр. </font>

# Содержимое проекта

<font color='#777778'>
Исследование выполнялось по плану, включающему в себя несколько этапов:
- 1. подготовка данных по классической модели ETL (извлечение, проверка, очистка и трансформация данных), что позволило минимизировать риск влияния вбросов и искажений и тем самым обеспечить качественную базу для последующих этапов обработки данных и анализа.
- 2. обработка данных
- 3. анализ данных
</font>


# Загрузка данных и знакомство с ними


In [4]:
# Загрузим необходимые библиотеки Python и данные датасета '/datasets/new_games.csv'.

import pandas as pd

try:
    # Ищем файл прямо в папке на компьютере
    df = pd.read_csv('new_games.csv')
except Exception as e:
    # Запасной вариант 
    df = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')
# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows 
pd.set_option('display.max_rows', 20)


In [5]:
# Выведем первые строки и результат метода info()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  str    
 1   Platform         16956 non-null  str    
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  str    
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  str    
 6   JP sales         16956 non-null  str    
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  str    
 10  Rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB


# Виды и типы данных для каждого столбца

Как видно на таблице выше, датасет содержит **16956 записей и 11 колонок**, в которых представлена информация о компьютерных играх, их продажах по регионам мира и различных рейтингах.

В том числе имеются следующие данные: 
•	Name — название игры
•	Platform — название платформы
•	Year of Release — год выпуска игры
•	Genre — жанр игры
•	NA sales — продажи в Северной Америке (в млн. проданных копий)
•	EU sales — продажи в Европе (в млн. проданных копий)
•	JP sales — продажи в Японии (в млн. проданных копий)
•	Other sales — продажи в других странах (в млн. проданных копий)
•	Critic Score — оценка критиков (от 0 до 100)
•	User Score — оценка пользователей (от 0 до 10)
•	Rating — рейтинг ESRB - Entertainment Software Rating Board (определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию)

 •	**Некорректные типы данных у некоторых столбцов**
Кроме того, некоторые данные имеют строковый формат: "EU sales", "JP sales", "User Score" и "Rating" вместо численного (например, float64), который более предпочтителен для цели дальнейших вычислений. "Year of Release" сохранен в формате float64 вместо более экономного и логичного integer.

 •	**Наличие пропусков в ряде столбцов**
Наибольшее количество пропусков содержится в столбцах Critic Score, User Score и Rating, что потребует более детального анализа и, возможно, поиска обходных путей, поскольку эти данные релевантны для дальнейшего анализа. В то же время, небольшая часть пропусков в столбце "Year of Release" отсеится сама собой, когда мы будем делать временной срез базы данных.

•	**Двусмысленные наименования столбцов**
В настоящее время некоторые наименования столбцов на английском и с использованием аббревиатур могут ввести в заблуждение русского пользователя. Например, "NA" - могло также означать "Australia & New Zealand".

In [6]:
# Выведем первые строки датафрейма на экран
df.head() 


,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


Как показывает таблица выше, датафрейм отсортирован по объему продаж отдельных игр в исторически лидирующем северноамериканском регионе. В топ-5 лидируют игры, разработанные для "динозавра" - легендарной домашней игровой консоли Nintendo (платформы WII, NES), и только разработчик Game Boу (Tetris, Покемоны), также присутствующий на рынке очень давно, смог составить им конкуренцию в топ-5. 


In [7]:
# Получим дополнительную информацию, описывающую количественные данные 
display(df.describe())

,Year of Release,NA sales,Other sales,Critic Score
count,16681.000000,16956.000000,16956.000000,8242.000000
mean,2006.485522,0.262023,0.047087,68.926717
std,5.873102,0.808654,0.185577,13.944565
min,1980.000000,0.000000,0.000000,13.000000
25%,2003.000000,0.000000,0.000000,60.000000
50%,2007.000000,0.080000,0.010000,71.000000
75%,2010.000000,0.240000,0.030000,79.000000
max,2016.000000,41.360000,10.570000,98.000000


- Как видно из таблицы выше, база данных содержит игры, **выпущенные с 1980 по 2016 гг.** 
- Как минимум, 25 % игр в базе относятся к Free-to-Play, т.е. они зарабатывают не на продаже лицензий, а на микротранзакциях внутри игры. Такие игры получили среднюю оценку критиков 60 %. 
- В то же время 50 % игр было проданы в количестве, не превышающем 80 тыс. лицензий. На примере североамериканского рынка видно, что разброс значений продаж гигантский. 
- По этому и другим показателям можно сделать промежуточный вывод о том, что рынок крайне фрагментированный: на одном конце находятся огромные студии со многомиллионными бюджетами на разработку и продвижение, что приносит им соответствующие доходы. 
- В то время как большое количество "indie" разработчиков имеют команды из нескольких человек и довольствуются "остатками с барского стола" (хотя и тут не обошлось без исключений типа Minecraft).

---

# Принятые допущения

При проведении данного исследования мы столкнулись с рядом **ограничений при доступе к данным**:
- В исходных данных отсутствовали стоимостные показатели продаж, что является ключевым показателем оценки объема рынка.
- Кроме того, по причине отсутствия информации о внутриигровых доходах оценка рынка может быть проведена исключительно на основе количества проданных лицензий, что не совсем корректно для free-to-play игр. 
- Разные игры были выпущены в разные временные периоды, однако, мы имеем в наличии только продажи с момента последнего релиза. Поэтому требуется осторожность при сравнении метрик по отдельным играм: например, игра, выпущенная в 2000 г., при прочих равных условиях акумулирует больше продаж к концу целевого периода (2013 г.), чем игра, которая вышла в 2010 г.
  
Вышеперечисленные факторы могут привести к перекосам в итоговых выводах, например, в пользу премиум-игр, североамериканского рынка, который исторически "выстрелил" первым, имеет собственную рейтинговую систему (ESRB) и т.д. 
Тогда как на текущем этапе лидируют именно мобильные игры, портативные консоли и азиатские рынки.

По поводу пропусков в отдельных столбцах следует отметить следующее: 
- Поскольку результаты данного исследования планируется использовать, прежде всего, в маркетинговых целях, а не для принятия бизнес-решений, незначительные погрешности в данных на данном этапе было решено игнорировать (например, количество пропусков в отдельных столбцах в пределах нормы - менее 1 %). 
Соответствующие строки будут удалены. 
- Напротив, пропуски в столбцах, используемых в вычислениях, будут заполнены средними значениями.


В связи с вышеупомянутыми ограничениями для цели дальнейшего анализа рекомендуем: 
- Исключить игры с бизнес-моделью Free-to-Play, так как метрика "Sales" для них не отражает реальный коммерческий успех из-за отсутствия данных по внутриигровым доходам. Кроме того, их присутствие в выборке привело бы к искажению ее общих показателей - агрегации.

- Где требуется, провести дополнительную сегментацию рынка на премиум-игры и бесплатные игры.

- Для цели сравнения популярности отдельных жанров использовать средние значения рейтингов (User score, Critic score). 


---

# Проверка ошибок в данных и их предобработка


## Названия, или метки, столбцов датафрейма

- Выведем названия всех столбцов датафрейма и проверим их стиль написания.
- Приведем все столбцы к стилю snake case: новые названия столбцов будут в нижнем регистре, а вместо пробелов — подчёркивания.

In [8]:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='str')

In [9]:
df.columns = df.columns.str.lower().str.replace(' ','_')
print(df.columns) 

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='str')


In [152]:
display(df.head())

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


# Типы данных


Как было указано выше, некоторые данные имеют строковый формат - EU sales, JP sales, User Score  - вместо численного (например, float64), который более предпочтителен для цели дальнейших вычислений. 
    Возможно, это произошло из-за использования неправильных символов, например, в случае со столбцами с данными о продажах причина вероятнее всего в отображении дробной части - в Европе используется запятая, а Python требует точки, или дело в лишних символах типа значка евро. Что касается столбцов по рейтингу, в данные могли попасть строковые символы: "нет данных", "n/a" и т.п.
Ниже преобразуем формат данных релевантных столбцов.

In [10]:
# Преобразуем столбцы "eu_sales" и "jp_sales" в численный формат и проверим результат
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce')
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  str    
 1   platform         16956 non-null  str    
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  str    
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       10152 non-null  str    
 10  rating           10085 non-null  str    
dtypes: float64(6), str(5)
memory usage: 1.4 MB


In [11]:
# Выборочно проверим, что изменилось (на примере eu_sales):
# Ищем строки, которые после конвертации стали пропусками (NaN)
display(df[df['eu_sales'].isna()])

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
446,Rhythm Heaven,DS,2008.0,Misc,0.55,NaN,1.93,0.13,83.0,9,E
802,Dead Rising,X360,2006.0,Action,1.16,NaN,0.08,0.20,85.0,7.6,M
1131,Prince of Persia: Warrior Within,PS2,2004.0,Action,0.54,NaN,0.00,0.22,83.0,8.5,M
1132,Far Cry 4,XOne,2014.0,Shooter,0.80,NaN,0.01,0.14,82.0,7.5,M
1394,Sonic Advance 3,GBA,2004.0,Platform,0.74,NaN,0.08,0.06,79.0,8.4,E
1612,Ratatouille,DS,2007.0,Action,0.49,NaN,0.00,0.14,NaN,NaN,NaN


In [13]:
# Преобразуем столбец "user_score" в численный формат, заменив строковые значения на пропуски, и проверим результат
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  str    
 1   platform         16956 non-null  str    
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  str    
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float64
 10  rating           10085 non-null  str    
dtypes: float64(7), str(4)
memory usage: 1.4 MB


---

## Наличие пропусков в данных

In [14]:
# Посчитаем количество пропусков в каждом столбце в абсолютных и относительных значениях.
empty_num = df.isna().sum()
empty_share = (empty_num * 100 / df.shape[0]).round(4) 
missing_data = pd.DataFrame({
    'Пропуски (шт.)': empty_num,
    'Доля пропусков (%)': empty_share
})
display(missing_data)


,Пропуски (шт.),Доля пропусков (%)
name,2,0.0118
platform,0,0.0000
year_of_release,275,1.6218
genre,2,0.0118
na_sales,0,0.0000
eu_sales,6,0.0354
jp_sales,4,0.0236
other_sales,0,0.0000
critic_score,8714,51.3918
user_score,9268,54.6591


Наибольшее количество пропусков выявлено в трех столбцах "critic_score", "user_score" и "rating". С большой долей вероятности можно предположить, что это системные пропуски, у которых есть исторические и технические причины: 
1. Игра была выпущена до эпохи интернета и сайтов оценок (появились только в 2000-х гг.).
2. Игра непопулярная или нишевая, поэтому крупные сайты ее просто проигнорировали (чтобы появился user_score, игру должно оценить определенное количество игроков, и т.д.).
3. Рейтинг ESRB платный, и если игра выходила только на внутреннем рынке Японии или Европы, разработчикам не было никакого смысла платить деньги за его получение.
4. Возможны и другие причины, например, потери при сборе данных, при загрузке и т.д.

<div class="alert alert-info">
<h2> Примечания по выбранным решения по пропускам в отдельных столбцах датафрейма <a class="tocSkip"> </h2>
    
Решения по пропускам в отдельных столбцах датафрейма обосновывались необходимостью обеспечить возможность использования этих столбцов в расчетах (в связи с особенностями python не допускаются символы, не соответствующие формату столбца).<br>
    - В столбце "rating", содержащем строковые значения, заменим NaN на "unknown".<br>
    - Удалим строки с пропусками существенной информации (название игры, год выпуска, жанр) в столбцах, которые потребуются в вычислениях.<br>
    - В столбцах "eu_sales" и "jp_sales" заполним пропуски средним с группировкой по году выпуска и платформе.<br>
    - Поменяем формат данных в столбце "year_of_release" на int64.
<div

In [17]:
# В столбце "rating", содержащем строковые значения, заменим NaN на "unknown".
df['rating'] = df['rating'].fillna('Unknown')

In [18]:
# Удалим строки с пропусками существенной информации (название игры, год выпуска, жанр) в столбцах, которые потребуются в вычислениях.
df = df.dropna(subset=['name', 'genre', 'year_of_release'])

# Поменяем формат данных в столбце "year_of_release" на int64.
df['year_of_release'] = df['year_of_release'].astype('int64')

In [19]:
# Для eu_sales и jp_sales заполним пропуски средним с группировкой по году выпуска и платформе.
mean_eu_sales = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].groupby(['year_of_release', 'platform'])['eu_sales'].mean()
mean_jp_sales = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].groupby(['year_of_release', 'platform'])['jp_sales'].mean()
eu_sales_table = pd.DataFrame(mean_eu_sales).reset_index()
jp_sales_table = pd.DataFrame(mean_jp_sales).reset_index()
print(f'Средний объем продаж на одну видеоигру в Европе в разрезе года выпуска и платформы (в шт.):')
display(eu_sales_table.sort_values(by='year_of_release', ascending=False).head(3))
print(f'Средний объем продаж на одну видеоигру в Японии в разрезе года выпуска и платформы (в шт.):')
display(jp_sales_table.sort_values(by='year_of_release', ascending=False).head(3))

Средний объем продаж на одну видеоигру в Европе в разрезе года выпуска и платформы (в шт.):


,year_of_release,platform,eu_sales
114,2013,PS4,0.690625
112,2013,PC,0.191538
110,2013,3DS,0.167849


Средний объем продаж на одну видеоигру в Японии в разрезе года выпуска и платформы (в шт.):


,year_of_release,platform,jp_sales
114,2013,PS4,0.058750
112,2013,PC,0.000000
110,2013,3DS,0.256559


<div class="alert alert-info">
<b> Пропуски в столбцах "eu_sales" и "jp_sales" были заполнены средним с группировкой по году выпуска и платформе.<br>

Поскольку на данном этапе исследования еще не требовалось фильтрации датафрейма по годам, мы применили фильтрацию только в рамках данной задачи (заполнения пропусков) - для большей наглядности результатов именно в интересуемый период. </b> 

---

# 3. Явные и неявные дубликаты в данных 

# Неявные дубликаты в данных


In [20]:
# Выведем список уникальных значений в столбце "genre" и отсортируем для наглядности.
unique_genre = sorted(df['genre'].unique())
print(unique_genre)


['ACTION', 'ADVENTURE', 'Action', 'Adventure', 'FIGHTING', 'Fighting', 'MISC', 'Misc', 'PLATFORM', 'PUZZLE', 'Platform', 'Puzzle', 'RACING', 'ROLE-PLAYING', 'Racing', 'Role-Playing', 'SHOOTER', 'SIMULATION', 'SPORTS', 'STRATEGY', 'Shooter', 'Simulation', 'Sports', 'Strategy']


In [22]:
# Промежуточный вывод: количество жанров очень велико в связи с их различным написанием в разных регистрах. Кроме того, на этапе анализа может быть целесообразным укрупнение жанров в новые категории для более удобного анализа.
# Приведем наименования жанров к нижнему регистру и отсортируем для наглядности.
df['genre'] = sorted(df['genre'].str.lower())
print(df['genre'].unique())

<StringArray>
[      'action',    'adventure',     'fighting',         'misc',
     'platform',       'puzzle',       'racing', 'role-playing',
      'shooter',   'simulation',       'sports',     'strategy']
Length: 12, dtype: str


In [23]:
# Выведем список уникальных значений в столбце "platform" и отсортируем для наглядности.
unique_platform = sorted(df['platform'].unique())
print(unique_platform)


['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG', 'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP', 'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WS', 'Wii', 'WiiU', 'X360', 'XB', 'XOne']


In [24]:
# Промежуточный вывод: нужно привести наименования платформ к верхнему регистру и далее - на этапе анализа - объединить платформы в укрупненные категории (например, все поколения одной платформы) для более удобного анализа.
df['platform'] = df['platform'].str.upper()

In [25]:
# Приведем наименования игр к нижнему регистру и уберем пробелы в начале и конце слов.
df['name'] = df['name'].str.lower().str.strip()

In [26]:
# Выведем список уникальных значений в столбце "year_of_release", отсортируем по возрастанию для наглядности.
# unique_year = sorted(df['year_of_release'].unique())
unique_year = df['year_of_release'].sort_values().unique().tolist()
print(unique_year)

[1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016]


Промежуточный вывод по годам: все чисто.

In [27]:
# Выведем список уникальных значений в столбце "rating".
unique_rating = df['rating'].unique().tolist()
print(unique_rating)

['E', 'Unknown', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']


<div class="alert alert-info">
<h2> Промежуточный вывод по столбцу "rating" <a class="tocSkip"> </h2>

Как видно, он содержит ряд значений, не принадлежащих к рейтингу ESRB.<br>
    
На самом деле, основных рейтингов всего шесть: "E" (Everyone), "E10+" (Everyone 10 and Up), "T" (Teen), "M" (Mature), "AO" (Adults Only) и "RP" (Rating pending). До 2018 г. также использовался рейтинг "EC" (Early childhood).<br>
    
Поскольку в будущем мы возможно будем делать группировку также и по рейтингу, необходимо привести ошибочные значения к норме. Предлагаемое решение выглядит следующим образом:<br>
- заменить значения "K-A" (kids-adults) на "Everyone"<br>
- заменить значение "EC" на "E".</b> 

In [28]:
# Посмотрим, распределение значений по столбцу "rating", чтобы принять решение, что делать с ошибочными значениями.
count_rating = df['rating'].value_counts()
share = round(count_rating * 100 / df['rating'].count(), 2)
rating_overview_df = pd.DataFrame({
    'Кол-во оценок': count_rating,
    'Доля (%)': share
})
display(rating_overview_df.sort_index(ascending=True))

,Кол-во оценок,Доля (%)
rating,,
AO,1,0.01
E,3968,23.79
E10+,1414,8.48
EC,8,0.05
K-A,3,0.02
M,1558,9.34
RP,1,0.01
T,2948,17.67
Unknown,6778,40.64


In [29]:
# Заменим ошибочно написанные значения рейтинга:
# - значение "K-A" (kids-adults) на "Everyone"
# - значение "EC" на "E"

df['rating'] = df['rating'].str.replace('K-A', 'E')
df['rating'] = df['rating'].str.replace('EC', 'E')


<div class="alert alert-info">
<h2> Промежуточный вывод по распределению рейтингов ESRB <a class="tocSkip"> </h2>
    
- Самую большую долю в датасете занимают игры с неуказанным рейтингом (Unknown - 40.64%). Это может быть связано с тем, что ESRB оценивает преимущественно игры для североамериканского рынка, либо отсутствуют данные по старым играм.<br> 
- Чтобы не терять этот огромный пласт данных при дальнейших группировках, мы оставим категорию "Unknown" как самостоятельную группу.<br>
    
- Среди известных рейтингов лидирует категория E («Для всех»), которая охватывает 23.86% от общего числа игр.<br> На втором месте идет рейтинг T («Для подростков») с долей 17.67%.

## Явные дубликаты в данных


In [30]:
# Проверим количество уникальных значений по столбцам и их долю в общем числе строк датафрейма (и округлим полученные значения).
unique_count = df.nunique() 
share_unique_count = (unique_count * 100 / df.shape[0]).round(2)
unique_values = pd.DataFrame({'unique_values_count': unique_count,
                        'share': share_unique_count})
print(f'Всего строк в датафрейме: {df.shape[0]}')
print(f'Количество и доля уникальных значений по столбцам (в %):')
display(unique_values)

Всего строк в датафрейме: 16679
Количество и доля уникальных значений по столбцам (в %):


,unique_values_count,share
name,11426,68.51
platform,31,0.19
year_of_release,37,0.22
genre,12,0.07
na_sales,401,2.40
eu_sales,307,1.84
jp_sales,244,1.46
other_sales,155,0.93
critic_score,81,0.49
user_score,95,0.57


In [31]:
# Проверим наличие явных дубликатов.
full_duplicates = df.duplicated().sum()
print(f'Количество полных дубликатов в таблице: {full_duplicates}')

Количество полных дубликатов в таблице: 235


In [33]:
# Удалим найденные дубликаты строк и обновляем индекс датафрейма (чтобы не было пропусков)
df = df.drop_duplicates()
df = df.reset_index(drop=True)

In [34]:
# Посчитаем количество удалённых строк в абсолютном и относительном значениях.
initial_row_num = 16956
new_row_num = df.shape[0]
deleted_num = initial_row_num - new_row_num
share_deleted = round((deleted_num * 100 / initial_row_num), 2)
unique_count_names = df['name'].nunique() 
print(f'Кол-во удаленных строк: {deleted_num}') 
print(f'Доля удаленных строк: {share_deleted}%') 
print(f'Новое кол-во строк в датафрейме: {new_row_num}') 
print(f'Уникальное кол-во игр: {unique_count_names}') 

Кол-во удаленных строк: 512
Доля удаленных строк: 3.02%
Новое кол-во строк в датафрейме: 16444
Уникальное кол-во игр: 11426


# Общий промежуточный вывод по предобработке данных


Как видно по обновленной таблице выше, после всех проделанных манипуляций по очистке данных у нас 16432 строк, но всего 11 421 уникальная игра.<br>

Для понимания причин нужно знать специфику игровой индустрии, где каждая игра может выпускаться одновременно на нескольких платформах.<br> Поэтому данные строки признаны легитимными и сохранены для дальнейшего анализа.<br>

После предобработки количество жанров пришло в норму (12 против 24 изначальных), общее количество платформ тоже соответствует среднему значению для отрасли (31). Также и оценки критиков и пользователей довольно плавные (обычно они идут по 100- или 10-балльной шкале).





# Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. 

In [35]:
# Подготовим срез данных за 2000-2013-ые гг. включительно в новом датафрейме df_actual
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].copy()
df_actual.info()
# Проверим полученные результаты
start_year = df_actual['year_of_release'].min()
end_year = df_actual['year_of_release'].max()
print(f'Первый год исследуемого периода:')
print(start_year)
print(f'Последний год исследуемого периода:')
print(end_year)

<class 'pandas.DataFrame'>
Index: 12781 entries, 0 to 16442
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12781 non-null  str    
 1   platform         12781 non-null  str    
 2   year_of_release  12781 non-null  int64  
 3   genre            12781 non-null  str    
 4   na_sales         12781 non-null  float64
 5   eu_sales         12776 non-null  float64
 6   jp_sales         12777 non-null  float64
 7   other_sales      12781 non-null  float64
 8   critic_score     7169 non-null   float64
 9   user_score       6483 non-null   float64
 10  rating           12781 non-null  str    
dtypes: float64(6), int64(1), str(4)
memory usage: 1.2 MB
Первый год исследуемого периода:
2000
Последний год исследуемого периода:
2013


---

# Категоризация данных
    
Проведем категоризацию данных:
- Разделим все игры по оценкам пользователей ("user_score") и выделим следующие категории: "высокая оценка" (от 8 до 10 включительно), "средняя оценка" (от 3 до 8, не включая правую границу интервала) и "низкая оценка" (от 0 до 3, не включая правую границу интервала).

In [36]:
df_actual['user_score_group'] = pd.cut(df_actual['user_score'], bins=[-0.1, 3, 8, 11], labels=["низкая оценка", "средняя оценка", "высокая оценка"], right=False)
# Выведем первые 10 строк для проверки.
display(df_actual[['name', 'user_score', 'user_score_group']].head(10))

,name,user_score,user_score_group
0,wii sports,8.0,высокая оценка
2,mario kart wii,8.3,высокая оценка
3,wii sports resort,8.0,высокая оценка
6,new super mario bros.,8.5,высокая оценка
7,wii play,6.6,средняя оценка
8,new super mario bros. wii,8.4,высокая оценка
10,nintendogs,NaN,NaN
11,mario kart ds,8.6,высокая оценка
13,wii fit,7.7,средняя оценка
14,kinect adventures!,6.3,средняя оценка


- Разделим все игры по оценкам критиков ("critic_score") и выделим такие категории: "высокая оценка" (от 80 до 100 включительно), "средняя оценка" (от 30 до 80, не включая правую границу интервала) и "низкая оценка" (от 0 до 30, не включая правую границу интервала).

In [37]:
df_actual['critic_score_group']  = pd.cut(df_actual['critic_score'], bins=[-0.1, 30, 80, 101], labels=["низкая оценка", "средняя оценка", "высокая оценка"], right=False)
# Выведем первые 10 строк для проверки.
display(df_actual[['name', 'critic_score', 'critic_score_group']].head(10))

,name,critic_score,critic_score_group
0,wii sports,76.0,средняя оценка
2,mario kart wii,82.0,высокая оценка
3,wii sports resort,80.0,высокая оценка
6,new super mario bros.,89.0,высокая оценка
7,wii play,58.0,средняя оценка
8,new super mario bros. wii,87.0,высокая оценка
10,nintendogs,NaN,NaN
11,mario kart ds,91.0,высокая оценка
13,wii fit,80.0,высокая оценка
14,kinect adventures!,61.0,средняя оценка


- После категоризации данных проверим результат: сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории.

In [38]:
# Посчитаем распределение нашего датафрейма по категориям пользовательского рейтинга, создадим таблицу для выдачи.
count_user_val = df_actual['user_score_group'].value_counts()
share_user_val = (df_actual['user_score_group'].value_counts(normalize=True) * 100).round(2)

grouped_user_table = pd.DataFrame({
    'Кол-во оценок': count_user_val,
    'Доля (%)': share_user_val
})

display(grouped_user_table.sort_index())


,Кол-во оценок,Доля (%)
user_score_group,,
низкая оценка,116,1.79
средняя оценка,4081,62.95
высокая оценка,2286,35.26


In [39]:
# Посчитаем распределение нашего датафрейма по категориям рейтинга критиков.
count_critic_val = df_actual['critic_score_group'].value_counts()
share_critic_val = (df_actual['critic_score_group'].value_counts(normalize=True) * 100).round(2)
grouped_critic_table = pd.DataFrame({
    'Кол-во оценок': count_critic_val,
    'Доля (%)': share_critic_val
})

display(grouped_critic_table.sort_index())

,Кол-во оценок,Доля (%)
critic_score_group,,
низкая оценка,55,0.77
средняя оценка,5422,75.63
высокая оценка,1692,23.60


<div class="alert alert-info">

Подавляющее большинство игр (более 98 %) получили средние и высокие оценки пользователей и критиков.


## Выделим топ-7 платформ по количеству игр, выпущенных за весь период (2000-2013).

- Разобъем задачу на подзадачи. Сначала сгруппируем данные по платформам, затем посчитаем количество игр и в конце отсортируем результат по убыванию, чтобы получить топ-7.

In [40]:
grouped_platform = df_actual['platform'].value_counts()
share_platform = (df_actual['platform'].value_counts(normalize = True)* 100).round(2)
name_platform = grouped_platform.index
platform_table = pd.DataFrame({'Название платформы': name_platform, 'Кол-во игр': grouped_platform, 'Доля игр в %': share_platform}, index=grouped_platform.index)
sorted_platform_table = platform_table.sort_values(by= 'Кол-во игр', ascending=False)

print(f'Топ-7 платформ по кол-ву игр:')
display(sorted_platform_table.head(7))


Топ-7 платформ по кол-ву игр:


,Название платформы,Кол-во игр,Доля игр в %
platform,,,
PS2,PS2,2127,16.64
DS,DS,2120,16.59
WII,WII,1275,9.98
PSP,PSP,1180,9.23
X360,X360,1121,8.77
PS3,PS3,1087,8.50
GBA,GBA,811,6.35


Верхние строчки рейтинга с минимальным отрывом друг от друга занимают мощные в технологическом плане платформы PS2 (2127 игры) и DS (2120 игр), суммарно насчитывающие более трети всех игр на рынке. За ними следуют платформы Wii (1275 игр), PSP (1126 игр), X360 (1121 игра) когда-то поразившие рынок своей инновационностью.

---

# Итоговые выводы
- В рамках исследования был реализован план работ по проверке, подготовке, фильтрации, обработке и анализу данных.
Исходная база данных довольно высокого качества, доля удаленных строк составила всего 3 %.<br><br> 
- Как видно по обновленной таблице выше, после всех проделанных манипуляций по очистке данных у нас 16432 строк, но всего 11 421 уникальная игра. Для понимания причин нужно знать специфику игровой индустрии, где каждая игра может выпускаться одновременно на нескольких платформах. Поэтому данные строки были признаны легитимными и сохранены для дальнейшего анализа.<br><br>
- После предобработки количество жанров пришло в норму (12 против 24 изначальных), общее количество платформ тоже соответствует среднему значению для отрасли (31). Также и оценки критиков и пользователей довольно плавные (обычно они идут по 100- или 10-балльной шкале).<br><br>
- Большинство игр получили средние и высокие оценки пользователей и критиков.<br><br>
- Верхние строчки рейтинга с минимальным отрывом друг от друга занимают мощные в технологическом плане платформы DS (2110 игр) и PS2 (2044 игры). Далее следуют платформы Wii (1261 игра) и PSP (1126 игр), когда-то поразившие рынок своей инновационностью.
